# 6. Main Regression Analysis

Confounders: How many `Confounders` are adjusted for.

Interaction: Whether a `Wealth x Burden` interaction term is present

| # | Confounders | Interaction |
|---|---|---|
| M1 | none | no |
| M2 | none | yes |
| M3 | chosen (Notebook 5, primary set) | no |
| M4 | chosen (Notebook 5, primary set) | yes |
| M5 | all confounder candidates | no |
| M6 | all confounder candidates | yes |

In [1]:
import time
import warnings

import numpy as np
import pandas as pd
import patsy
from scipy import stats
from statsmodels.miscmodels.ordinal_model import OrderedModel

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 100)

## Data

In [2]:
df_depression = pd.read_csv("../resources/depression_dataset.csv")
df_anxiety = pd.read_csv("../resources/anxiety_dataset.csv")

# ordinal outcomes used as-is: Depression 0-4 (PHQ-9 bands), Anxiety 0-3 (GAD-7 bands)
for label, data, col in [("Depression", df_depression, "Depression"), ("Anxiety", df_anxiety, "Anxiety")]:
    print(f"{label}: n={len(data)}")
    print(data[col].value_counts().sort_index())
    print()

Depression: n=4887
Depression
0    3475
1    1177
2     173
3      57
4       5
Name: count, dtype: int64

Anxiety: n=4887
Anxiety
0    3544
1    1122
2     175
3      46
Name: count, dtype: int64



## 2. Mean-centering Wealth and Burden, and checking Wealth's linearity assumption

Both Socioeconomic Status (1=Poorest ... 5=Richest) and Cardiometabolic Burden (0-3) are
mean-centered and entered as single linear terms. Confounder sets are otherwise the same three
used throughout this project, taken as-is from Notebook 5:

- **None** (M1-M2) -- wealth and burden only.
- **Chosen** (M3-M4) -- Notebook 5's primary set, minus Age at first cohabitation (not yet in the
  analytic file -- Notebook 5 Section 7). 7 covariates.
- **All candidates** (M5-M6) -- the 20 available candidates, minus Insurance (0/16 insured women
  land above band 1 on either outcome -- checked directly below) and minus the same
  not-yet-available Age at first cohabitation.

In [3]:
burden_mean, wealth_mean = {}, {}
for label, data in [("Depression", df_depression), ("Anxiety", df_anxiety)]:
    bm = data["Cardiometabolic Burden"].mean()
    wm = data["Socioeconomic Status"].mean()
    data["Burden_c"] = data["Cardiometabolic Burden"] - bm
    data["Wealth_c"] = data["Socioeconomic Status"] - wm
    burden_mean[label], wealth_mean[label] = bm, wm
    print(f"{label}: mean burden = {bm:.4f}, mean wealth = {wm:.4f}")

Depression: mean burden = 0.3413, mean wealth = 3.1232
Anxiety: mean burden = 0.3413, mean wealth = 3.1232


In [4]:
CHOSEN = ["Age", "Division", "Residence", "Religion", "Education", "Occupation", "Partner occupation"]

ALL_CANDIDATES_RAW = [
    "Education", "Occupation", "Partner occupation", "Age", "Division", "Residence", "Religion",
    "Children", "Family size", "Household Autonomy", "Financial Decision-Making", "IPV Attitude",
    "Insurance", "Internet", "Contraceptive", "Abortion", "Pregnant", "Menopause",
    "Sexual activity", "Postpartum",
]

for label, data, col in [("Depression", df_depression, "Depression"), ("Anxiety", df_anxiety, "Anxiety")]:
    print(label)
    print(pd.crosstab(data["Insurance"], data[col]))
    print()

ALL_CANDIDATES = [c for c in ALL_CANDIDATES_RAW if c != "Insurance"]
print(f"chosen: {len(CHOSEN)} covariates -- {CHOSEN}")
print(f"all candidates: {len(ALL_CANDIDATES)} covariates (Insurance dropped -- see crosstabs above)")

Depression
Depression     0     1    2   3  4
Insurance                         
0.0         3465  1171  173  57  5
1.0           10     6    0   0  0

Anxiety
Anxiety       0     1    2   3
Insurance                     
0.0        3530  1120  175  46
1.0          14     2    0   0

chosen: 7 covariates -- ['Age', 'Division', 'Residence', 'Religion', 'Education', 'Occupation', 'Partner occupation']
all candidates: 19 covariates (Insurance dropped -- see crosstabs above)


**Linearity check for Wealth.** The same idea Notebook 5/`docs/analysis-notes.md` already
applied to Burden (checking a linear trend against a fully flexible categorical coding) is applied
to Wealth here: fit the no-confounder, no-interaction model with Wealth as `C(...)` (4 dummy
contrasts) instead of linear, and compare by likelihood-ratio test. A non-significant result means
the linear term isn't losing much fit relative to letting every wealth category have its own
coefficient -- i.e. the linearity assumption used in every model below is reasonable.

## 3. Weighted ordinal model

In [5]:
class WeightedOrderedModel(OrderedModel):
    def __init__(self, endog, exog, weights, **kwargs):
        self._wts = np.asarray(weights)
        super().__init__(endog, exog, **kwargs)

    def loglikeobs(self, params):
        return super().loglikeobs(params) * self._wts

    def loglike(self, params):
        return np.sum(self.loglikeobs(params))


WEALTH = "Q('Wealth_c')"
BURDEN = "Q('Burden_c')"


def build_formula(outcome_col, confounders, interaction):
    wealth_burden = f"{WEALTH} * {BURDEN}" if interaction else f"{WEALTH} + {BURDEN}"
    terms = [wealth_burden] + [f"C(Q('{c}'))" for c in confounders]
    return f"{outcome_col} ~ " + " + ".join(terms)


def fit_formula(data, outcome_col, formula):
    """Fit a weighted, cluster-robust ordinal model for an arbitrary patsy formula."""
    y, X = patsy.dmatrices(formula, data, return_type="dataframe")
    X = X.drop(columns=["Intercept"])
    endog = data[outcome_col].astype(int)

    model = WeightedOrderedModel(endog, X, data["Sampling weight"].values, distr="logit")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = model.fit(
            method="bfgs", maxiter=1000, disp=False,
            cov_type="cluster", cov_kwds={"groups": data["PSU"].values},
        )
    res.formula = formula
    res.converged = res.mle_retvals.get("converged", None)
    return res


def fit_model(data, outcome_col, confounders, interaction, label):
    res = fit_formula(data, outcome_col, build_formula(outcome_col, confounders, interaction))
    res.label, res.n_confounders, res.interaction = label, len(confounders), interaction
    return res


def full_table(res):
    """Every term in the model -- wealth, burden, interaction, every confounder dummy, and the
    ordinal thresholds -- as an odds-ratio table. Thresholds are cutpoints between outcome
    categories, not covariate effects, but are included here for completeness."""
    terms = list(res.params.index)
    ci = res.conf_int().loc[terms]
    return pd.DataFrame({
        "term": terms,
        "OR": np.exp(res.params[terms]).values,
        "CI low": np.exp(ci[0]).values,
        "CI high": np.exp(ci[1]).values,
        "p": res.pvalues[terms].values,
    })


def show_model(data, outcome_col, confounders, interaction, label):
    res = fit_model(data, outcome_col, confounders, interaction, label)
    print(f"--- {label} (n={len(data)}, confounders={len(confounders)}, "
          f"interaction={interaction}, converged={res.converged}) ---")
    print(full_table(res).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    return res


results = {}

## 4. Wealth linearity check

In [6]:
linearity_rows = []
for outcome, data, col in [("Depression", df_depression, "Depression"), ("Anxiety", df_anxiety, "Anxiety")]:
    linear_formula = f"{col} ~ {WEALTH} + {BURDEN}"
    categorical_formula = f"{col} ~ C(Q('Socioeconomic Status')) + {BURDEN}"

    res_linear = fit_formula(data, col, linear_formula)
    res_categorical = fit_formula(data, col, categorical_formula)

    lr_stat = 2 * (res_categorical.llf - res_linear.llf)
    df_diff = len(res_categorical.params) - len(res_linear.params)
    p = stats.chi2.sf(lr_stat, df_diff)
    linearity_rows.append({"outcome": outcome, "LR chi2": round(lr_stat, 3), "df": int(df_diff), "p": round(p, 4)})

linearity_check = pd.DataFrame(linearity_rows)
linearity_check

,outcome,LR chi2,df,p
0,Depression,2.988,3,0.3935
1,Anxiety,2.063,3,0.5594


## 5. Fitting all twelve models, one at a time

Each cell below fits exactly one model and prints its full coefficient table. Results are stashed
in `results` as they go, for the omnibus tests and comparison table in Sections 6-7.

### Depression -- M1 (no confounders, no interaction)

In [7]:
# Crude/unadjusted model: Wealth and Burden as independent linear predictors, nothing else.
results[("Depression", "M1")] = show_model(
    df_depression, "Depression", [], False, "Depression M1"
)

--- Depression M1 (n=4887, confounders=0, interaction=False, converged=True) ---
         term     OR  CI low  CI high      p
Q('Wealth_c') 0.9046  0.8558   0.9562 0.0004
Q('Burden_c') 1.1583  1.0397   1.2905 0.0077
          0/1 2.5085  2.3017   2.7338 0.0000
          1/2 2.0901  1.9385   2.2535 0.0000
          2/3 1.2866  1.0441   1.5855 0.0180
          3/4 2.4420  1.7048   3.4982 0.0000


### Depression -- M2 (no confounders, with interaction)

In [8]:
# Same as M1 but adds the Wealth x Burden product term -- tests whether burden's association with the outcome depends on wealth, with no confounders adjusted for.
results[("Depression", "M2")] = show_model(
    df_depression, "Depression", [], True, "Depression M2"
)

--- Depression M2 (n=4887, confounders=0, interaction=True, converged=True) ---
                       term     OR  CI low  CI high      p
              Q('Wealth_c') 0.9043  0.8555   0.9559 0.0004
              Q('Burden_c') 1.1450  1.0282   1.2751 0.0136
Q('Wealth_c'):Q('Burden_c') 1.0394  0.9586   1.1269 0.3494
                        0/1 2.5224  2.3148   2.7487 0.0000
                        1/2 2.0902  1.9386   2.2536 0.0000
                        2/3 1.2867  1.0442   1.5856 0.0180
                        3/4 2.4431  1.7053   3.5001 0.0000


### Depression -- M3 (chosen confounders, no interaction)

In [9]:
# Adds Notebook 5's primary (DAG-selected) confounder set to the main-effects-only model -- the project's main-effects "adjusted" model.
results[("Depression", "M3")] = show_model(
    df_depression, "Depression", CHOSEN, False, "Depression M3"
)

--- Depression M3 (n=4887, confounders=7, interaction=False, converged=True) ---
                           term     OR  CI low  CI high      p
               C(Q('Age'))[T.2] 1.3402  1.1095   1.6189 0.0024
               C(Q('Age'))[T.3] 1.4522  1.1913   1.7702 0.0002
          C(Q('Division'))[T.2] 1.1340  0.8276   1.5539 0.4340
          C(Q('Division'))[T.3] 0.7614  0.5698   1.0174 0.0653
          C(Q('Division'))[T.4] 1.1661  0.8737   1.5563 0.2968
          C(Q('Division'))[T.5] 0.8024  0.5945   1.0830 0.1502
          C(Q('Division'))[T.6] 0.8835  0.6555   1.1909 0.4163
          C(Q('Division'))[T.7] 1.2646  0.9341   1.7121 0.1287
          C(Q('Division'))[T.8] 0.8321  0.5969   1.1599 0.2780
         C(Q('Residence'))[T.2] 0.8896  0.7348   1.0772 0.2309
          C(Q('Religion'))[T.2] 0.6701  0.5106   0.8795 0.0039
         C(Q('Education'))[T.1] 0.8938  0.7157   1.1160 0.3216
         C(Q('Education'))[T.2] 0.8632  0.6913   1.0778 0.1940
         C(Q('Education'))[T.3] 0.674

### Depression -- M4 (chosen confounders, with interaction)

In [10]:
# Adds the same primary confounder set to the interaction model -- the project's primary moderation test, confounder-adjusted.
results[("Depression", "M4")] = show_model(
    df_depression, "Depression", CHOSEN, True, "Depression M4"
)

--- Depression M4 (n=4887, confounders=7, interaction=True, converged=True) ---
                           term     OR  CI low  CI high      p
               C(Q('Age'))[T.2] 1.3397  1.1092   1.6181 0.0024
               C(Q('Age'))[T.3] 1.4513  1.1908   1.7688 0.0002
          C(Q('Division'))[T.2] 1.1356  0.8291   1.5556 0.4282
          C(Q('Division'))[T.3] 0.7590  0.5683   1.0138 0.0619
          C(Q('Division'))[T.4] 1.1665  0.8740   1.5569 0.2957
          C(Q('Division'))[T.5] 0.8011  0.5938   1.0809 0.1467
          C(Q('Division'))[T.6] 0.8862  0.6573   1.1947 0.4279
          C(Q('Division'))[T.7] 1.2633  0.9332   1.7100 0.1304
          C(Q('Division'))[T.8] 0.8342  0.5984   1.1628 0.2847
         C(Q('Residence'))[T.2] 0.8900  0.7352   1.0774 0.2321
          C(Q('Religion'))[T.2] 0.6714  0.5114   0.8814 0.0041
         C(Q('Education'))[T.1] 0.8928  0.7148   1.1151 0.3175
         C(Q('Education'))[T.2] 0.8617  0.6899   1.0763 0.1896
         C(Q('Education'))[T.3] 0.6732

### Depression -- M5 (all confounder candidates, no interaction)

In [11]:
# Main-effects-only model adjusted for every confounder candidate Notebook 5 screened (minus Insurance and the not-yet-available Age at first cohabitation) -- a "kitchen sink" robustness check on the main effects.
results[("Depression", "M5")] = show_model(
    df_depression, "Depression", ALL_CANDIDATES, False, "Depression M5"
)

--- Depression M5 (n=4887, confounders=19, interaction=False, converged=True) ---
                                  term     OR  CI low  CI high      p
                C(Q('Education'))[T.1] 0.9347  0.7468   1.1700 0.5558
                C(Q('Education'))[T.2] 0.8835  0.7025   1.1111 0.2896
                C(Q('Education'))[T.3] 0.6548  0.4863   0.8818 0.0053
             C(Q('Occupation'))[T.1.0] 1.0144  0.8698   1.1830 0.8553
       C(Q('Partner occupation'))[T.2] 0.6064  0.4136   0.8890 0.0104
       C(Q('Partner occupation'))[T.3] 1.6769  0.3687   7.6265 0.5035
                      C(Q('Age'))[T.2] 1.4108  1.1334   1.7561 0.0021
                      C(Q('Age'))[T.3] 1.5166  1.1708   1.9646 0.0016
                 C(Q('Division'))[T.2] 1.1373  0.8320   1.5546 0.4198
                 C(Q('Division'))[T.3] 0.7450  0.5557   0.9988 0.0491
                 C(Q('Division'))[T.4] 1.1729  0.8748   1.5726 0.2864
                 C(Q('Division'))[T.5] 0.8129  0.6000   1.1013 0.1812
        

### Depression -- M6 (all confounder candidates, with interaction)

In [12]:
# The same kitchen-sink confounder set, with the Wealth x Burden interaction added -- a robustness check on the moderation test itself.
results[("Depression", "M6")] = show_model(
    df_depression, "Depression", ALL_CANDIDATES, True, "Depression M6"
)

--- Depression M6 (n=4887, confounders=19, interaction=True, converged=True) ---
                                  term     OR  CI low  CI high      p
                C(Q('Education'))[T.1] 0.9335  0.7456   1.1686 0.5480
                C(Q('Education'))[T.2] 0.8815  0.7007   1.1089 0.2813
                C(Q('Education'))[T.3] 0.6530  0.4852   0.8788 0.0049
             C(Q('Occupation'))[T.1.0] 1.0166  0.8718   1.1854 0.8339
       C(Q('Partner occupation'))[T.2] 0.6069  0.4146   0.8882 0.0102
       C(Q('Partner occupation'))[T.3] 1.6427  0.3654   7.3843 0.5175
                      C(Q('Age'))[T.2] 1.4122  1.1343   1.7582 0.0020
                      C(Q('Age'))[T.3] 1.5186  1.1719   1.9679 0.0016
                 C(Q('Division'))[T.2] 1.1383  0.8330   1.5555 0.4161
                 C(Q('Division'))[T.3] 0.7425  0.5541   0.9951 0.0463
                 C(Q('Division'))[T.4] 1.1729  0.8748   1.5726 0.2865
                 C(Q('Division'))[T.5] 0.8113  0.5991   1.0988 0.1766
         

### Anxiety -- M1 (no confounders, no interaction)

In [13]:
# Crude/unadjusted model: Wealth and Burden as independent linear predictors, nothing else.
results[("Anxiety", "M1")] = show_model(
    df_anxiety, "Anxiety", [], False, "Anxiety M1"
)

--- Anxiety M1 (n=4887, confounders=0, interaction=False, converged=True) ---
         term     OR  CI low  CI high      p
Q('Wealth_c') 0.9031  0.8558   0.9532 0.0002
Q('Burden_c') 1.1892  1.0694   1.3224 0.0014
          0/1 2.7236  2.4917   2.9770 0.0000
          1/2 2.0554  1.9051   2.2175 0.0000
          2/3 1.6292  1.3159   2.0171 0.0000


### Anxiety -- M2 (no confounders, with interaction)

In [14]:
# Same as M1 but adds the Wealth x Burden product term -- tests whether burden's association with the outcome depends on wealth, with no confounders adjusted for.
results[("Anxiety", "M2")] = show_model(
    df_anxiety, "Anxiety", [], True, "Anxiety M2"
)

--- Anxiety M2 (n=4887, confounders=0, interaction=True, converged=True) ---
                       term     OR  CI low  CI high      p
              Q('Wealth_c') 0.9027  0.8552   0.9529 0.0002
              Q('Burden_c') 1.1801  1.0600   1.3137 0.0025
Q('Wealth_c'):Q('Burden_c') 1.0274  0.9541   1.1063 0.4737
                        0/1 2.7341  2.5005   2.9896 0.0000
                        1/2 2.0554  1.9052   2.2175 0.0000
                        2/3 1.6293  1.3159   2.0172 0.0000


### Anxiety -- M3 (chosen confounders, no interaction)

In [15]:
# Adds Notebook 5's primary (DAG-selected) confounder set to the main-effects-only model -- the project's main-effects "adjusted" model.
results[("Anxiety", "M3")] = show_model(
    df_anxiety, "Anxiety", CHOSEN, False, "Anxiety M3"
)

--- Anxiety M3 (n=4887, confounders=7, interaction=False, converged=True) ---
                           term     OR  CI low  CI high      p
               C(Q('Age'))[T.2] 1.6887  1.3585   2.0992 0.0000
               C(Q('Age'))[T.3] 2.0341  1.6045   2.5786 0.0000
          C(Q('Division'))[T.2] 1.1981  0.8518   1.6851 0.2990
          C(Q('Division'))[T.3] 0.7154  0.5215   0.9814 0.0378
          C(Q('Division'))[T.4] 1.1522  0.8263   1.6067 0.4035
          C(Q('Division'))[T.5] 0.7072  0.5167   0.9679 0.0305
          C(Q('Division'))[T.6] 0.9444  0.6608   1.3499 0.7538
          C(Q('Division'))[T.7] 1.2338  0.8889   1.7125 0.2092
          C(Q('Division'))[T.8] 0.9010  0.6409   1.2666 0.5485
         C(Q('Residence'))[T.2] 0.7290  0.6026   0.8819 0.0011
          C(Q('Religion'))[T.2] 0.5751  0.4247   0.7787 0.0003
         C(Q('Education'))[T.1] 0.8693  0.6978   1.0829 0.2115
         C(Q('Education'))[T.2] 0.8226  0.6521   1.0378 0.0996
         C(Q('Education'))[T.3] 0.5382  

### Anxiety -- M4 (chosen confounders, with interaction)

In [16]:
# Adds the same primary confounder set to the interaction model -- the project's primary moderation test, confounder-adjusted.
results[("Anxiety", "M4")] = show_model(
    df_anxiety, "Anxiety", CHOSEN, True, "Anxiety M4"
)

--- Anxiety M4 (n=4887, confounders=7, interaction=True, converged=True) ---
                           term     OR  CI low  CI high      p
               C(Q('Age'))[T.2] 1.6881  1.3578   2.0986 0.0000
               C(Q('Age'))[T.3] 2.0333  1.6042   2.5772 0.0000
          C(Q('Division'))[T.2] 1.1994  0.8525   1.6874 0.2966
          C(Q('Division'))[T.3] 0.7136  0.5201   0.9790 0.0365
          C(Q('Division'))[T.4] 1.1518  0.8257   1.6067 0.4053
          C(Q('Division'))[T.5] 0.7058  0.5157   0.9659 0.0295
          C(Q('Division'))[T.6] 0.9461  0.6615   1.3532 0.7615
          C(Q('Division'))[T.7] 1.2333  0.8886   1.7117 0.2099
          C(Q('Division'))[T.8] 0.9030  0.6420   1.2701 0.5577
         C(Q('Residence'))[T.2] 0.7289  0.6026   0.8817 0.0011
          C(Q('Religion'))[T.2] 0.5758  0.4250   0.7800 0.0004
         C(Q('Education'))[T.1] 0.8687  0.6971   1.0824 0.2098
         C(Q('Education'))[T.2] 0.8220  0.6515   1.0370 0.0983
         C(Q('Education'))[T.3] 0.5377  0

### Anxiety -- M5 (all confounder candidates, no interaction)

In [17]:
# Main-effects-only model adjusted for every confounder candidate Notebook 5 screened (minus Insurance and the not-yet-available Age at first cohabitation) -- a "kitchen sink" robustness check on the main effects.
results[("Anxiety", "M5")] = show_model(
    df_anxiety, "Anxiety", ALL_CANDIDATES, False, "Anxiety M5"
)

--- Anxiety M5 (n=4887, confounders=19, interaction=False, converged=True) ---
                                  term     OR  CI low  CI high      p
                C(Q('Education'))[T.1] 0.9191  0.7354   1.1487 0.4583
                C(Q('Education'))[T.2] 0.8713  0.6872   1.1049 0.2556
                C(Q('Education'))[T.3] 0.5774  0.4075   0.8182 0.0020
             C(Q('Occupation'))[T.1.0] 0.8826  0.7440   1.0470 0.1518
       C(Q('Partner occupation'))[T.2] 0.6300  0.4227   0.9390 0.0232
       C(Q('Partner occupation'))[T.3] 1.4214  0.2304   8.7677 0.7048
                      C(Q('Age'))[T.2] 1.6612  1.2846   2.1481 0.0001
                      C(Q('Age'))[T.3] 1.8864  1.3811   2.5765 0.0001
                 C(Q('Division'))[T.2] 1.1694  0.8303   1.6471 0.3705
                 C(Q('Division'))[T.3] 0.6862  0.4985   0.9445 0.0209
                 C(Q('Division'))[T.4] 1.1203  0.7997   1.5696 0.5089
                 C(Q('Division'))[T.5] 0.6887  0.5005   0.9477 0.0220
           

### Anxiety -- M6 (all confounder candidates, with interaction)

In [18]:
# The same kitchen-sink confounder set, with the Wealth x Burden interaction added -- a robustness check on the moderation test itself.
results[("Anxiety", "M6")] = show_model(
    df_anxiety, "Anxiety", ALL_CANDIDATES, True, "Anxiety M6"
)

--- Anxiety M6 (n=4887, confounders=19, interaction=True, converged=True) ---
                                  term     OR  CI low  CI high      p
                C(Q('Education'))[T.1] 0.9181  0.7345   1.1477 0.4531
                C(Q('Education'))[T.2] 0.8702  0.6863   1.1034 0.2511
                C(Q('Education'))[T.3] 0.5765  0.4070   0.8165 0.0019
             C(Q('Occupation'))[T.1.0] 0.8845  0.7454   1.0497 0.1601
       C(Q('Partner occupation'))[T.2] 0.6310  0.4244   0.9383 0.0229
       C(Q('Partner occupation'))[T.3] 1.3995  0.2201   8.8998 0.7218
                      C(Q('Age'))[T.2] 1.6629  1.2856   2.1509 0.0001
                      C(Q('Age'))[T.3] 1.8890  1.3831   2.5800 0.0001
                 C(Q('Division'))[T.2] 1.1710  0.8310   1.6503 0.3670
                 C(Q('Division'))[T.3] 0.6844  0.4971   0.9421 0.0200
                 C(Q('Division'))[T.4] 1.1198  0.7989   1.5696 0.5113
                 C(Q('Division'))[T.5] 0.6875  0.4997   0.9459 0.0213
            

## 6. Omnibus test for the interaction

Nested likelihood-ratio test: each interaction model against its no-interaction counterpart at the
same confounder tier. With Wealth and Burden both linear, the interaction is a single product
term, so every comparison below has 1 degree of freedom.

In [19]:
def lr_test(res_restricted, res_full):
    lr_stat = 2 * (res_full.llf - res_restricted.llf)
    df_diff = len(res_full.params) - len(res_restricted.params)
    p = stats.chi2.sf(lr_stat, df_diff)
    return lr_stat, df_diff, p


rows = []
for outcome in ["Depression", "Anxiety"]:
    for restricted, full in [("M1", "M2"), ("M3", "M4"), ("M5", "M6")]:
        lr, dfd, p = lr_test(results[(outcome, restricted)], results[(outcome, full)])
        rows.append({"outcome": outcome, "comparison": f"{restricted} -> {full}",
                     "LR chi2": round(lr, 3), "df": int(dfd), "p": round(p, 4)})

lr_summary = pd.DataFrame(rows)
lr_summary

,outcome,comparison,LR chi2,df,p
0,Depression,M1 -> M2,1.124,1,0.2891
1,Depression,M3 -> M4,1.332,1,0.2485
2,Depression,M5 -> M6,1.187,1,0.2758
3,Anxiety,M1 -> M2,0.550,1,0.4582
4,Anxiety,M3 -> M4,0.733,1,0.3920
5,Anxiety,M5 -> M6,0.846,1,0.3577


## 7. Model comparison table

`EPV` is a binary-outcome rule of thumb and doesn't translate directly to an ordinal model with
multiple thresholds, so it's replaced here with observations per parameter (N / number of
estimated parameters, thresholds included) -- a cruder, more general sparseness check.

In [20]:
rows = []
for outcome, data in [("Depression", df_depression), ("Anxiety", df_anxiety)]:
    for name in ["M1", "M2", "M3", "M4", "M5", "M6"]:
        res = results[(outcome, name)]
        k = len(res.params)
        rows.append({
            "outcome": outcome, "model": name, "confounders": res.n_confounders,
            "interaction": res.interaction, "parameters": k,
            "AIC": round(res.aic, 1), "BIC": round(res.bic, 1),
            "N per parameter": round(len(data) / k, 2),
        })

model_summary = pd.DataFrame(rows)
model_summary

,outcome,model,confounders,interaction,parameters,AIC,BIC,N per parameter
0,Depression,M1,0,False,6,7366.8,7405.7,814.50
1,Depression,M2,0,True,7,7367.7,7413.1,698.14
2,Depression,M3,7,False,23,7319.4,7468.8,212.48
3,Depression,M4,7,True,24,7320.1,7476.0,203.62
4,Depression,M5,19,False,45,7311.8,7604.1,108.60
5,Depression,M6,19,True,46,7312.6,7611.4,106.24
6,Anxiety,M1,0,False,5,7061.7,7094.2,977.40
7,Anxiety,M2,0,True,6,7063.1,7102.1,814.50
8,Anxiety,M3,7,False,22,6926.4,7069.3,222.14
9,Anxiety,M4,7,True,23,6927.7,7077.1,212.48


## 8. What this notebook does not settle

- Wealth's linearity check (Section 4) should be read before trusting the trend coefficients
  above -- if it comes back significant, the single Wealth term is masking a non-monotonic
  pattern.
- Age at first cohabitation is still missing from the "chosen" set -- not yet in the analytic
  file (Notebook 5 Section 7).
- The proportional-odds assumption (that wealth/burden/confounder effects are the same across all
  outcome cutoffs) is still not tested here.
- No established design-based (survey-weighted, stratified) ordinal tool exists on either the
  Python or R side to cross-check these numbers against.
- `docs/analysis-notes.md`'s 11-covariate primary confounder set still predates and doesn't match
  Notebook 5's per-candidate DAG output -- unreconciled, independent of everything in this
  notebook.